# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library, referencing all data entities using their `@id` fields as defined in the Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets and their fields and IDs.

We use the Croissant `@id` references for all entities below.

In [ ]:
# List all available record sets and their fields by @id
print("Available record sets and their fields:")
record_sets = []
for record_set in metadata.record_sets:
    print(f"- Record set name: {record_set.name}, @id: {record_set.id}")
    record_sets.append(record_set.id)
    if hasattr(record_set, "fields"):
        for field in record_set.fields:
            print(f"    - Field: {field.name}, @id: {field.id}, type: {field.data_type}")
        print('')
if not record_sets:
    print('No record sets defined in the schema or they are not discoverable by mlcroissant.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

For this dataset, we load all discovered record sets using their `@id`. If the record set(s) include(s) fields like regression results, socio-demographic values, or responses, you'll see them in the DataFrame preview below.

In [ ]:
dataframes = {}

if record_sets:
    for record_set_id in record_sets:
        try:
            print(f"Loading records for record_set '@id': {record_set_id}")
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns for @{record_set_id}: {df.columns.tolist()}")
            display(df.head())
        except Exception as e:
            print(f"Failed to load record set {record_set_id}: {e}")
    if dataframes:
        # Example: get the first record set and show its columns
        first_id = list(dataframes.keys())[0]
        print(dataframes[first_id].columns.tolist())
        display(dataframes[first_id].head())
    else:
        print('No record sets could be loaded.')
else:
    print('No record sets to extract from.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing, and grouping. For this section, ensure field names and group columns refer to their `@id` (not display names), as discovered above.

**Below, refer to a numeric field using its `@id`.**

In [ ]:
# Select a numeric field @id and a group field @id, as listed in Data Overview above
# (Replace these example @ids with actual ones from your dataset if different)
example_record_set_id = record_sets[0] if record_sets else None
example_numeric_field_id = None
example_group_field_id = None

# Automatically pick the first numeric column if detected
if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    numeric_columns = df.select_dtypes(include=['number']).columns
    if len(numeric_columns) > 0:
        example_numeric_field_id = numeric_columns[0]
    group_candidates = [col for col in df.columns if df[col].dtype == 'O']
    if group_candidates:
        example_group_field_id = group_candidates[0]

    if example_numeric_field_id:
        print(f"Using numeric field '@id': {example_numeric_field_id}")
        print(f"Using group field '@id': {example_group_field_id}")

        # Filter records
        threshold = df[example_numeric_field_id].mean() if df[example_numeric_field_id].notna().any() else 0
        filtered_df = df[df[example_numeric_field_id] > threshold]
        print(f"Filtered records with {example_numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize numeric field
        norm_field = f"{example_numeric_field_id}_normalized"
        filtered_df[norm_field] = (
            filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()
        ) / filtered_df[example_numeric_field_id].std() if filtered_df[example_numeric_field_id].std() else 0
        print(f"Normalized {example_numeric_field_id} for filtered records:")
        display(filtered_df[[example_numeric_field_id, norm_field]].head())

        # Grouping (if group field is valid)
        if example_group_field_id and example_group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(example_group_field_id)[example_numeric_field_id].mean().reset_index()
            print(f"Grouped data by {example_group_field_id} (mean of {example_numeric_field_id}):")
            display(grouped_df.head())
    else:
        print('No numeric field detected for EDA in this record set.')
else:
    print('No suitable record set data for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using the detected `@id` fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and example_numeric_field_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[example_numeric_field_id].dropna(), bins=15)
    plt.title(f"Distribution of {example_numeric_field_id}")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If possible, visualise relationship to group field
    if example_group_field_id and example_group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=example_group_field_id, y=example_numeric_field_id, data=df)
        plt.title(f"{example_numeric_field_id} by {example_group_field_id}")
        plt.xlabel(example_group_field_id)
        plt.ylabel(example_numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to programmatically browse, extract, and analyze structured data from a Croissant-based dataset using `mlcroissant`.

- All dataset entities were referenced by their Croissant `@id`.
- You explored available record sets and fields.
- Data was normalized and grouped based on chosen (numeric and categorical) fields.
- Data distributions were visualized using Seaborn/Matplotlib.

This workflow is extensible to downstream statistical modeling or integration with additional Croissant-conformant datasets.